# Preparazione dataset: WMH e ATLAS 2.0 per nnU-Net v2

Questo notebook trasforma i dati grezzi **read-only** in due dataset nnU-Net v2 pronti al planning,
con split train/validation/test riproducibili. I volumi sorgente non vengono mai modificati.

Il filo conduttore è la **risoluzione spaziale**: ogni decisione (target spacing, resampling,
sopravvivenza delle lesioni piccole, stratificazione degli split) viene misurata prima di essere presa.

## Indice delle operazioni

| Step | Operazione | Output |
| --- | --- | --- |
| 1 | Configurazione e layout delle cartelle di output | `data/processed/` |
| 2 | Indicizzazione dei casi e appaiamento immagine/label | lista casi |
| 3 | Manifest geometrico: spacing, shape, FOV, orientamento, volumi lesione | `manifests/*.csv` |
| 4 | Analisi della risoluzione spaziale e anisotropia | figure in `qc/` |
| 5 | Target spacing nnU-Net e shape stimate dopo resampling | tabella |
| 6 | QC e politica delle label (WMH label 2, ATLAS due varianti) | report |
| 7 | Split test / 5-fold CV stratificati, senza leakage | `splits/*.json` |
| 8 | Materializzazione nel formato nnU-Net raw | `data/processed/nnUNet_raw/` |
| 9 | `dataset.json` e `splits_final.json` | file nnU-Net |
| 10 | QC finale: integrità, geometria, overlay visivo | figure + assert |
| 11 | Comandi nnU-Net successivi | - |

## Principi seguiti

- **Nessun resampling manuale.** Il resampling lo fa nnU-Net durante il preprocessing, una volta sola.
  Resamplare qui e poi lasciare che nnU-Net resampli di nuovo introduce una doppia interpolazione che
  degrada le lesioni piccole. Qui *misuriamo* la risoluzione e *prevediamo* cosa farà il planner.
- **Immagini collegate, label riscritte.** Le immagini sono grandi e non vanno modificate: si usano
  symlink (costo zero su disco). Le label sono piccole e vanno normalizzate (dtype, valori, geometria):
  vengono riscritte.
- **Il test set esce dal training set.** I casi di test non finiscono in `imagesTr`, quindi non entrano
  nel fingerprint nnU-Net e non contaminano né le statistiche di normalizzazione né la scelta del piano.

## Step 1 - Configurazione

Tutti i parametri modificabili stanno in questa cella. Il resto del notebook non contiene costanti nascoste.

In [ ]:
from pathlib import Path
import json, os, shutil, sys, time
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

PROJECT = Path('/home/ccantini/lesion_detection_project')
CFG = json.loads((PROJECT / 'configs' / 'datasets.json').read_text())
WMH_ROOT = Path(CFG['datasets']['WMH']['path'])
ATLAS_ROOT = Path(CFG['datasets']['ATLAS']['path'])

# --- Output (tutto sotto data/, come richiesto) --------------------------------
PROC = PROJECT / 'data' / 'processed'
MANIFEST_DIR = PROC / 'manifests'
SPLIT_DIR = PROC / 'splits'
QC_DIR = PROC / 'qc'
RAW_OUT = PROC / 'nnUNet_raw'          # dataset nnU-Net materializzati
NNUNET_RAW = PROJECT / 'nnUNet' / 'nnUNet_raw'          # riceverà symlink
NNUNET_PREPROCESSED = PROJECT / 'nnUNet' / 'nnUNet_preprocessed'
NNUNET_RESULTS = PROJECT / 'nnUNet' / 'nnUNet_results'

# --- Parametri di preparazione -------------------------------------------------
SEED = 42
N_FOLDS = 5
TEST_FRACTION = {'WMH': 0.20, 'ATLAS': 0.15}   # 0.0 = niente holdout, solo CV
VOLUME_BINS = 3                                 # bin di carico lesionale per la stratificazione

DATASET_IDS = {'WMH': 501, 'ATLAS': 502}

WMH_USE_T1 = False          # True -> secondo canale T1 gia' registrato su FLAIR
WMH_LABEL2_POLICY = 'exclude'   # 'exclude' (2->0) | 'ignore' (ignore label) | 'merge' (2->1)

MATERIALIZE_IMAGES = 'symlink'  # 'symlink' (0 byte) | 'copy'
FORCE_RECOMPUTE = False         # True per rigenerare i manifest ignorando la cache

rng = np.random.default_rng(SEED)
for d in (MANIFEST_DIR, SPLIT_DIR, QC_DIR, RAW_OUT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS):
    d.mkdir(parents=True, exist_ok=True)

print('WMH   :', WMH_ROOT, '->', WMH_ROOT.exists())
print('ATLAS :', ATLAS_ROOT, '->', ATLAS_ROOT.exists())
print('output:', PROC)
print('nnunetv2 disponibile:', __import__('importlib').util.find_spec('nnunetv2') is not None)

## Step 2 - Indicizzazione dei casi

I due dataset hanno convenzioni di naming completamente diverse, quindi servono due funzioni di discovery
che producono però la **stessa struttura record**. Da qui in poi il codice è comune.

**WMH** - un caso per cartella, naming piatto per sito:

```
Utrecht_0/
  Utrecht_0_FLAIR_BiasCorr.nii.gz                  <- canale 0000 (baseline)
  Utrecht_0_T1w_BiasCorr_2FLAIR_Aligned.nii.gz     <- canale 0001 opzionale, gia' su griglia FLAIR
  Utrecht_0_wmh.nii.gz                             <- label (0/1/2)
```
Gli altri file (`_BiasField`, `_T1w_BiasCorr` non allineato, `.mat`) sono sottoprodotti della correzione
di bias e **non** vanno usati. Il gruppo di stratificazione è il **sito/scanner**
(`Amsterdam_GE1T5`, `Amsterdam_GE3T`, `Singapore`, `Utrecht`).

**ATLAS 2.0** - struttura BIDS, il gruppo di stratificazione è la **coorte** (`R001`...`R052`):

```
Training/R001/sub-r001s001/ses-1/anat/
  sub-r001s001_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz                       <- canale 0000
  ..._label-L_desc-T1lesion_mask.nii.gz                                         <- label (valori 0/1)
  ..._label-L_desc-T1lesion_mask_value2.nii.gz                                  <- stessa maschera, valori 0/2
```
Il `Testing/` ufficiale contiene **solo immagini**, senza label: non è utilizzabile per valutare
localmente, va tenuto da parte per l'eventuale submission.

In [ ]:
def wmh_cases():
    """Un record per soggetto WMH. Il gruppo di stratificazione e' il sito/scanner."""
    out = []
    for flair in sorted(WMH_ROOT.rglob('*_FLAIR_BiasCorr.nii.gz')):
        stem = flair.name[:-len('_FLAIR_BiasCorr.nii.gz')]
        label = flair.with_name(stem + '_wmh.nii.gz')
        t1 = flair.with_name(stem + '_T1w_BiasCorr_2FLAIR_Aligned.nii.gz')
        if not label.exists():
            print('  [skip] label mancante per', stem)
            continue
        out.append(dict(dataset='WMH', case_id=stem,
                        group=stem.rsplit('_', 1)[0],          # sito/scanner
                        image=flair,
                        image2=t1 if t1.exists() else None,
                        label=label,
                        source_split='train'))
    return out


def atlas_cases():
    """Un record per soggetto ATLAS. Il gruppo di stratificazione e' la coorte Rxxx."""
    out = []
    for source_split, folder in (('train', 'Training'), ('official_test', 'Testing')):
        for image in sorted((ATLAS_ROOT / folder).rglob('*_T1w.nii.gz')):
            label = image.with_name(image.name.replace('_T1w.nii.gz',
                                                       '_label-L_desc-T1lesion_mask.nii.gz'))
            out.append(dict(dataset='ATLAS', case_id=image.name.split('_ses-')[0],
                            group=image.parents[3].name,        # R001, R002, ...
                            image=image,
                            image2=None,
                            label=label if label.exists() else None,
                            source_split=source_split))
    return out


CASES = {'WMH': wmh_cases(), 'ATLAS': atlas_cases()}

for name, cs in CASES.items():
    ids = [c['case_id'] for c in cs]
    assert len(ids) == len(set(ids)), f'{name}: case_id duplicati'
    n_lab = sum(c['label'] is not None for c in cs)
    n_t1 = sum(c['image2'] is not None for c in cs)
    print(f'{name:6s} casi={len(cs):4d}  con label={n_lab:4d}  con 2o canale={n_t1:4d}  '
          f'gruppi={len(set(c["group"] for c in cs))}')
    print('        source_split:', pd.Series([c['source_split'] for c in cs]).value_counts().to_dict())

## Step 3 - Manifest geometrico

Per ogni caso leggiamo l'header NIfTI (economico, `mmap`) e la label (richiede la decompressione completa).
Registriamo:

- **spacing** `sx, sy, sz` in mm e **shape** in voxel;
- **FOV** in mm per asse (`shape * spacing`) - serve a capire se i volumi coprono lo stesso campo di vista;
- **orientamento** (codici assiali, es. `LPS`/`RAS`) - se non è uniforme serve una riorientazione;
- **anisotropia** = `max(spacing) / min(spacing)`, la soglia che nnU-Net usa a 3.0;
- **volume voxel** in mm³ e **volume lesione in mm³** - il volume va calcolato in mm³, non in voxel,
  altrimenti dataset con spacing diverso non sono confrontabili;
- **`geom_ok`**: shape e affine di immagine e label coincidono (requisito nnU-Net);
- **valori label presenti**, per scoprire codifiche anomale.

La lettura di tutte le label ATLAS richiede alcuni minuti: il manifest viene messo in cache su CSV e
ricalcolato solo se `FORCE_RECOMPUTE = True`.

In [ ]:
def case_metadata(case):
    img = nib.load(str(case['image']), mmap=True)
    zoom = tuple(float(v) for v in img.header.get_zooms()[:3])
    shape = tuple(int(v) for v in img.shape[:3])
    voxel_vol = float(np.prod(zoom))

    rec = dict(dataset=case['dataset'], case_id=case['case_id'], group=case['group'],
               source_split=case['source_split'],
               sx=zoom[0], sy=zoom[1], sz=zoom[2],
               nx=shape[0], ny=shape[1], nz=shape[2],
               fov_x=shape[0] * zoom[0], fov_y=shape[1] * zoom[1], fov_z=shape[2] * zoom[2],
               orientation=''.join(nib.aff2axcodes(img.affine)),
               voxel_vol_mm3=voxel_vol,
               anisotropy=max(zoom) / min(zoom),
               has_ch1=case['image2'] is not None,
               ch1_geom_ok=np.nan, geom_ok=np.nan,
               label_values='', n_lesion_vox=np.nan, lesion_mm3=np.nan,
               n_other_vox=np.nan, other_mm3=np.nan)

    if case['image2'] is not None:
        ch1 = nib.load(str(case['image2']), mmap=True)
        rec['ch1_geom_ok'] = bool(tuple(int(v) for v in ch1.shape[:3]) == shape
                                  and np.allclose(ch1.affine, img.affine, atol=1e-3))

    if case['label'] is not None:
        lab = nib.load(str(case['label']))
        data = np.asanyarray(lab.dataobj)
        rec['geom_ok'] = bool(tuple(int(v) for v in lab.shape[:3]) == shape
                              and np.allclose(lab.affine, img.affine, atol=1e-3))
        values = np.unique(data)
        rec['label_values'] = '/'.join(f'{v:g}' for v in values)
        # binarizzazione robusta: > 0 e non == 1, perche' esistono maschere codificate 0/0.01
        rec['n_lesion_vox'] = int((data > 0).sum() if case['dataset'] == 'ATLAS'
                                  else (np.rint(data) == 1).sum())
        rec['lesion_mm3'] = rec['n_lesion_vox'] * voxel_vol
        if case['dataset'] == 'WMH':
            rec['n_other_vox'] = int((np.rint(data) == 2).sum())
            rec['other_mm3'] = rec['n_other_vox'] * voxel_vol
    return rec


def build_manifest(name):
    path = MANIFEST_DIR / f'{name}_cases.csv'
    if path.exists() and not FORCE_RECOMPUTE:
        print(f'{name}: manifest da cache {path}')
        return pd.read_csv(path)
    rows, cases, t0 = [], CASES[name], time.time()
    for i, c in enumerate(cases, 1):
        rows.append(case_metadata(c))
        if i % 100 == 0 or i == len(cases):
            print(f'  {name}: {i}/{len(cases)}  ({time.time() - t0:.0f}s)', flush=True)
    df = pd.DataFrame(rows)
    df.to_csv(path, index=False)
    print(f'{name}: manifest scritto in {path}')
    return df


MANIFEST = {name: build_manifest(name) for name in ('WMH', 'ATLAS')}
MANIFEST['ATLAS'].head(3)

## Step 4 - Analisi della risoluzione spaziale

Questo è il punto centrale. Le due domande a cui rispondiamo:

1. **Quanto è eterogenea la risoluzione dentro ogni dataset?** Se lo è molto, il resampling a una spacing
   comune deforma alcuni casi molto più di altri.
2. **Quanto è anisotropo il voxel?** Un voxel molto allungato lungo z significa che le lesioni sottili
   sono già mal campionate nel dato grezzo, e nessun preprocessing può recuperare quell'informazione.

In [ ]:
def resolution_summary(df, by='group'):
    g = df.groupby(by)
    out = pd.DataFrame({
        'n': g.size(),
        'sx': g['sx'].agg(lambda s: f'{s.median():.4g}'),
        'sy': g['sy'].agg(lambda s: f'{s.median():.4g}'),
        'sz': g['sz'].agg(lambda s: f'{s.median():.4g}'),
        'shape_mediana': g.apply(lambda d: f"{d.nx.median():.0f}x{d.ny.median():.0f}x{d.nz.median():.0f}",
                                 include_groups=False),
        'fov_mm_mediano': g.apply(lambda d: f"{d.fov_x.median():.0f}x{d.fov_y.median():.0f}x{d.fov_z.median():.0f}",
                                  include_groups=False),
        'voxel_mm3': g['voxel_vol_mm3'].median().round(3),
        'anisotropia': g['anisotropy'].median().round(2),
        'orient': g['orientation'].agg(lambda s: '/'.join(sorted(set(s)))),
    })
    return out


for name, df in MANIFEST.items():
    print(f'\n===== {name}: risoluzione per gruppo =====')
    display(resolution_summary(df))
    print('spacing distinti nel dataset:',
          df[['sx', 'sy', 'sz']].round(4).drop_duplicates().shape[0])
    print('orientamenti distinti     :', sorted(df.orientation.unique()))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for row, (name, color) in enumerate([('WMH', 'tab:blue'), ('ATLAS', 'tab:orange')]):
    df = MANIFEST[name]
    for ax, col, lbl in zip(axes[row], ['sx', 'sy', 'sz'],
                            ['spacing x (in-plane)', 'spacing y (in-plane)', 'spacing z (through-plane)']):
        ax.hist(df[col], bins=30, color=color)
        ax.set_title(f'{name} - {lbl}')
        ax.set_xlabel('mm')
        ax.set_ylabel('n casi')
        ax.axvline(df[col].median(), ls='--', c='k', lw=1,
                   label=f'mediana {df[col].median():.3f}')
        ax.legend(fontsize=8)
plt.suptitle('Distribuzione della risoluzione spaziale per asse', fontsize=13)
plt.tight_layout()
plt.savefig(QC_DIR / 'spacing_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for name, color in [('WMH', 'tab:blue'), ('ATLAS', 'tab:orange')]:
    ax.hist(MANIFEST[name]['anisotropy'], bins=30, alpha=.65, label=name, color=color)
ax.axvline(3.0, ls='--', c='r', label='soglia nnU-Net = 3.0')
ax.set_xlabel('anisotropia = max(spacing) / min(spacing)')
ax.set_ylabel('n casi')
ax.legend()
ax.set_title('Anisotropia del voxel')
plt.tight_layout()
plt.savefig(QC_DIR / 'anisotropy.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 5 - Target spacing nnU-Net e shape dopo resampling

nnU-Net sceglie la target spacing con una regola precisa (`ExperimentPlanner.determine_fullres_target_spacing`):

1. `target = mediana` degli spacing, asse per asse;
2. identifica l'asse **peggiore** (spacing mediano più alto);
3. applica un override **solo se entrambe** le condizioni sono vere:
   - `spacing[peggiore] > 3 * max(spacing degli altri assi)`,
   - `shape[peggiore] * 3 < min(shape degli altri assi)`;
4. in quel caso usa il **10º percentile** degli spacing su quell'asse (recupera risoluzione dove i dati
   la hanno), senza mai scendere sotto lo spacing degli altri assi.

Qui la riproduciamo per **prevedere** il piano prima di lanciare il preprocessing (che dura ore) e per
stimare le shape risultanti. Le shape reali saranno leggermente inferiori perché nnU-Net croppa prima
alla regione non nulla; la stima resta valida per capire l'ordine di grandezza.

Separatamente si applica la scelta del **metodo di resampling**: se l'anisotropia supera 3, nnU-Net
resampla il piano in-plane e l'asse z in modo separato (`do_separate_z`), per non spalmare le lesioni
tra fette adiacenti. Le due decisioni sono indipendenti: un dataset può non ricevere l'override della
target spacing e usare comunque il resampling separato.

In [ ]:
ANISO_THRESHOLD = 3.0   # nnunetv2.experiment_planning ... ANISO_THRESHOLD


def nnunet_target_spacing(spacings, shapes):
    """Riproduce determine_fullres_target_spacing di nnU-Net v2."""
    spacings = np.asarray(spacings, dtype=float)
    shapes = np.asarray(shapes, dtype=float)
    target = np.percentile(spacings, 50, axis=0)
    target_size = np.percentile(shapes, 50, axis=0)

    worst = int(np.argmax(target))
    others = [i for i in range(3) if i != worst]
    other_sp = [target[i] for i in others]
    other_sz = [target_size[i] for i in others]

    aniso_spacing = bool(target[worst] > ANISO_THRESHOLD * max(other_sp))
    aniso_voxels = bool(target_size[worst] * ANISO_THRESHOLD < min(other_sz))

    override = False
    if aniso_spacing and aniso_voxels:
        cand = float(np.percentile(spacings[:, worst], 10))
        if cand < max(other_sp):
            cand = max(max(other_sp), cand) + 1e-5
        target[worst] = cand
        override = True

    diag = dict(asse_peggiore=worst,
                spacing_mediano=np.round(np.percentile(spacings, 50, axis=0), 4).tolist(),
                shape_mediana=target_size.astype(int).tolist(),
                cond_spacing=f'{target[worst] if override else np.percentile(spacings,50,0)[worst]:.4g} > '
                             f'{ANISO_THRESHOLD}*{max(other_sp):.4g}={ANISO_THRESHOLD*max(other_sp):.4g} -> {aniso_spacing}',
                cond_voxel=f'{target_size[worst]:.0f}*{ANISO_THRESHOLD} < {min(other_sz):.0f} -> {aniso_voxels}',
                override_applicato=override)
    return target, diag


PLAN = {}
for name, df in MANIFEST.items():
    train_like = df[df.source_split == 'train']
    sp = train_like[['sx', 'sy', 'sz']].to_numpy()
    sh = train_like[['nx', 'ny', 'nz']].to_numpy()
    target, diag = nnunet_target_spacing(sp, sh)
    resampled = np.round(sh * sp / target).astype(int)
    do_separate_z = bool(max(target) / min(target) > ANISO_THRESHOLD)
    PLAN[name] = dict(target_spacing=target, diag=diag, do_separate_z=do_separate_z)

    print(f'===== {name} =====')
    print(f'  spacing mediano   : {np.round(np.median(sp,0),4).tolist()} mm')
    print(f'  asse peggiore     : {"xyz"[diag["asse_peggiore"]]}')
    print(f'  condizione spacing: {diag["cond_spacing"]}')
    print(f'  condizione voxel  : {diag["cond_voxel"]}')
    print(f'  override applicato: {diag["override_applicato"]}')
    print(f'  >> TARGET SPACING : {np.round(target,4).tolist()} mm')
    print(f'  >> resampling separato lungo z (anisotropia>3): {do_separate_z}')
    print(f'  shape stimata dopo resampling: mediana {np.median(resampled,0).astype(int).tolist()}, '
          f'max {resampled.max(0).tolist()}')
    print(f'  voxel per volume: prima {np.median(sh.prod(1))/1e6:.1f}M -> dopo {np.median(resampled.prod(1))/1e6:.1f}M')
    print()

### Sopravvivenza delle lesioni piccole al resampling

Una lesione sopravvive al resampling se il suo volume resta apprezzabile rispetto al **volume del voxel
target**. Calcoliamo per ogni caso quanti voxel occuperà la lesione dopo il resampling: i casi che
scendono sotto poche decine di voxel sono quelli su cui il Dice sarà instabile, e sono anche quelli da
distribuire uniformemente fra i fold (motivo per cui più avanti stratifichiamo per carico lesionale).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, name in zip(axes, ['WMH', 'ATLAS']):
    df = MANIFEST[name]
    df = df[(df.source_split == 'train') & df.lesion_mm3.notna()]
    target_vox_mm3 = float(np.prod(PLAN[name]['target_spacing']))
    vox_after = df.lesion_mm3 / target_vox_mm3
    ax.hist(np.log10(vox_after.clip(lower=1)), bins=40, color='tab:green')
    ax.axvline(np.log10(100), ls='--', c='r', label='100 voxel')
    ax.set_xlabel('log10(voxel di lesione dopo resampling)')
    ax.set_ylabel('n casi')
    ax.set_title(f'{name}: voxel target {target_vox_mm3:.3f} mm3')
    ax.legend(fontsize=8)
    print(f'{name}: volume lesione mm³  min={df.lesion_mm3.min():.0f}  p10={df.lesion_mm3.quantile(.1):.0f}  '
          f'mediana={df.lesion_mm3.median():.0f}  p90={df.lesion_mm3.quantile(.9):.0f}  max={df.lesion_mm3.max():.0f}')
    print(f'      casi < 100 voxel dopo resampling: {(vox_after < 100).sum()} / {len(df)}')
    print(f'      casi con lesione vuota          : {(df.n_lesion_vox == 0).sum()}')
plt.suptitle('Dimensione della lesione in voxel alla risoluzione target')
plt.tight_layout()
plt.savefig(QC_DIR / 'lesion_size_after_resampling.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 6 - QC e politica delle label

Due questioni lasciate aperte dalla documentazione del progetto, che qui vengono risolte con i dati.

### ATLAS: le due varianti di maschera

Ogni caso ha `..._mask.nii.gz` e `..._mask_value2.nii.gz`. La cella sotto verifica su un campione che
**il supporto binario è identico** e che cambia solo il valore codificato (1 contro 2): la seconda è una
comodità per sovrapporre maschere, non una segmentazione diversa. Usiamo `_mask.nii.gz` e ignoriamo
l'altra.

⚠️ Esiste inoltre almeno una maschera ATLAS codificata con valore `0.01` invece di `1`. Un
`label == 1` la trasformerebbe silenziosamente in un caso completamente vuoto. Per questo la
binarizzazione ATLAS usa `> 0`.

### WMH: il significato della label 2

Le label WMH hanno tre valori: `0` background, `1` iperintensità della sostanza bianca, `2` **altra
patologia** (tipicamente infarti), che nella challenge ufficiale è **esclusa dalla valutazione**.
Le tre politiche disponibili:

| `WMH_LABEL2_POLICY` | Mappatura | Quando usarla |
| --- | --- | --- |
| `exclude` (default) | `2 -> 0` | Coerente con la valutazione ufficiale. Il modello impara a non segmentare le altre patologie. |
| `ignore` | `2 -> ignore label nnU-Net` | Più rigoroso: su quei voxel la loss non viene calcolata, quindi non si insegna né "lesione" né "background" su tessuto ambiguo. |
| `merge` | `2 -> 1` | **Sconsigliata.** Insegna al modello a chiamare WMH anche gli infarti, e rende il Dice non confrontabile con la letteratura. |

In [ ]:
# --- ATLAS: le due varianti di maschera sono lo stesso supporto? -----------------
sample = [c for c in CASES['ATLAS'] if c['label'] is not None][:8]
print('ATLAS - confronto mask vs mask_value2 (campione):')
for c in sample:
    m2 = c['label'].with_name(c['label'].name.replace('_mask.nii.gz', '_mask_value2.nii.gz'))
    if not m2.exists():
        print(f"  {c['case_id']}: variante value2 assente")
        continue
    a = np.asanyarray(nib.load(str(c['label'])).dataobj)
    b = np.asanyarray(nib.load(str(m2)).dataobj)
    print(f"  {c['case_id']}: valori {np.unique(a).tolist()} vs {np.unique(b).tolist()}  "
          f"stesso supporto binario = {np.array_equal(a > 0, b > 0)}")

# --- codifiche anomale -----------------------------------------------------------
print('\nValori di label osservati:')
for name, df in MANIFEST.items():
    vc = df[df.label_values != ''].label_values.value_counts()
    print(f'  {name}: {vc.to_dict()}')
    anomali = df[(df.label_values != '') & (~df.label_values.isin(['0/1', '0/1/2', '0']))]
    for _, r in anomali.iterrows():
        print(f'    ANOMALO -> {r.case_id}: valori {r.label_values} '
              f'({r.n_lesion_vox:.0f} voxel positivi recuperati con la soglia > 0)')

# --- WMH: quanto pesa la label 2 -------------------------------------------------
w = MANIFEST['WMH']
print(f"\nWMH: casi con label 2 presente: {(w.n_other_vox > 0).sum()} / {len(w)}")
print(f"     volume 'altra patologia' mm³: mediana {w.other_mm3.median():.0f}, max {w.other_mm3.max():.0f}")
print(f"     politica scelta: WMH_LABEL2_POLICY = '{WMH_LABEL2_POLICY}'")

# --- geometria immagine/label ----------------------------------------------------
print('\nGeometria immagine/label coerente:')
for name, df in MANIFEST.items():
    sub = df[df.geom_ok.notna()]
    print(f'  {name}: {int(sub.geom_ok.sum())}/{len(sub)} ok')
    assert sub.geom_ok.all(), f'{name}: geometria immagine/label non coerente'
    if df.has_ch1.any():
        s2 = df[df.ch1_geom_ok.notna()]
        print(f'  {name}: secondo canale allineato {int(s2.ch1_geom_ok.sum())}/{len(s2)}')
        assert s2.ch1_geom_ok.all(), f'{name}: secondo canale non allineato'

## Step 7 - Split train / validation / test

### Cosa significa "validation" con nnU-Net

nnU-Net non usa un singolo split train/val: fa **5-fold cross-validation**, e in ogni fold il quinto di
dati escluso *è* il validation set. Quindi produciamo:

- un **test set** tenuto completamente fuori (holdout), mai visto né dal training né dal planning;
- **5 fold** sui dati restanti; ogni fold definisce la sua coppia train/validation.

Questo è più informativo di un singolo split 80/20 su dataset di queste dimensioni (99 casi per WMH),
e per usarlo basta scrivere il file `splits_final.json` che nnU-Net legge invece di generarne uno proprio.

### Stratificazione

Lo split casuale puro è rischioso qui, per due motivi misurati negli step precedenti:

1. **Eterogeneità di sito/coorte.** WMH ha 4 siti con scanner e risoluzioni diverse; ATLAS ha 33 coorti.
   Se un fold pesca quasi solo un sito, il Dice di validation misura la generalizzazione a quel sito,
   non la qualità del modello.
2. **Carico lesionale molto sbilanciato.** Il volume di lesione copre tre ordini di grandezza. I casi
   con lesioni minuscole dominano la varianza del Dice: vanno spalmati uniformemente.

Quindi stratifichiamo su una **chiave composta `gruppo | terzile di volume lesionale`**. Le classi troppo
rare per essere divise vengono unite in una classe `RARE` (altrimenti `StratifiedKFold` fallisce).

### Assenza di leakage

Ogni soggetto compare in un solo caso e in un solo insieme, quindi non c'è leakage per soggetto.
Le verifiche esplicite sono nella cella di controllo.

### Il test ufficiale ATLAS

Le 300 immagini di `Testing/` non hanno label: **non** possono servire da test locale. Vengono
materializzate a parte, solo per un'eventuale inferenza da sottomettere.

In [ ]:
from sklearn.model_selection import StratifiedKFold, train_test_split


def strat_key(df, n_bins=VOLUME_BINS):
    """gruppo | terzile di carico lesionale."""
    ranks = df['lesion_mm3'].rank(method='first')
    bins = pd.qcut(ranks, n_bins, labels=[f'v{i}' for i in range(n_bins)])
    return (df['group'].astype(str) + '|' + bins.astype(str)).rename('strat')


def merge_rare(keys, min_count):
    """Unisce in 'RARE' le classi con meno di min_count membri."""
    vc = keys.value_counts()
    rare = set(vc[vc < min_count].index)
    if rare:
        print(f'    {len(rare)} classi con <{min_count} membri unite in RARE '
              f'({int(keys.isin(rare).sum())} casi)')
    return keys.where(~keys.isin(rare), 'RARE')


def build_split(name):
    df = MANIFEST[name]
    pool = df[df.source_split == 'train'].reset_index(drop=True)
    keys = strat_key(pool)

    # 1) holdout di test
    frac = TEST_FRACTION[name]
    if frac > 0:
        k_test = merge_rare(keys, 2)
        tr_idx, te_idx = train_test_split(np.arange(len(pool)), test_size=frac,
                                          random_state=SEED, stratify=k_test)
    else:
        tr_idx, te_idx = np.arange(len(pool)), np.array([], dtype=int)

    dev = pool.iloc[tr_idx].reset_index(drop=True)
    test_ids = sorted(pool.iloc[te_idx].case_id.tolist())

    # 2) 5-fold CV sul resto
    k_cv = merge_rare(strat_key(dev), N_FOLDS)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    folds = [{'train': sorted(dev.case_id.iloc[a].tolist()),
              'val': sorted(dev.case_id.iloc[b].tolist())}
             for a, b in skf.split(np.zeros(len(dev)), k_cv)]

    official = sorted(df[df.source_split == 'official_test'].case_id.tolist())
    split = dict(dataset=name, seed=SEED, n_folds=N_FOLDS, test_fraction=frac,
                 stratified_on=f'group | terzile lesion_mm3 (bins={VOLUME_BINS})',
                 development=sorted(dev.case_id.tolist()),
                 internal_test=test_ids,
                 official_test_unlabeled=official,
                 folds=folds)
    (SPLIT_DIR / f'{name}_split.json').write_text(json.dumps(split, indent=2))
    return split


SPLITS = {}
for name in ('WMH', 'ATLAS'):
    print(f'--- {name} ---')
    SPLITS[name] = build_split(name)
    s = SPLITS[name]
    print(f'  development (train+val CV): {len(s["development"])}')
    print(f'  internal test (con label) : {len(s["internal_test"])}')
    print(f'  official test (no label)  : {len(s["official_test_unlabeled"])}')
    print(f'  fold sizes train/val      : ' +
          ', '.join(f'{len(f["train"])}/{len(f["val"])}' for f in s['folds']))

### Verifica degli split

Controlliamo esplicitamente le proprietà che devono valere, con `assert`: se una salta, il notebook
si ferma invece di produrre silenziosamente un dataset sbagliato.

In [ ]:
for name, s in SPLITS.items():
    df = MANIFEST[name].set_index('case_id')
    dev, test = set(s['development']), set(s['internal_test'])
    pool = set(MANIFEST[name].query('source_split == "train"').case_id)

    assert dev.isdisjoint(test), f'{name}: development e test si sovrappongono'
    assert dev | test == pool, f'{name}: casi persi o duplicati fra development e test'
    for i, f in enumerate(s['folds']):
        tr, va = set(f['train']), set(f['val'])
        assert tr.isdisjoint(va), f'{name} fold {i}: train e val si sovrappongono'
        assert tr | va == dev, f'{name} fold {i}: non copre il development set'
        assert va.isdisjoint(test), f'{name} fold {i}: val contiene casi di test'
    assert set().union(*[set(f['val']) for f in s['folds']]) == dev, \
        f'{name}: le validation dei fold non partizionano il development set'
    print(f'{name}: tutti i controlli di consistenza superati')

    # bilanciamento per gruppo e per carico lesionale
    rows = []
    for part, ids in [('test', s['internal_test'])] + \
                     [(f'val{i}', f['val']) for i, f in enumerate(s['folds'])]:
        sub = df.loc[sorted(ids)]
        r = {'n': len(sub), 'lesion_mm3_mediana': round(sub.lesion_mm3.median())}
        r.update((sub.group.value_counts(normalize=True) * 100).round(0).astype(int)
                 .add_suffix('%').to_dict())
        rows.append(pd.Series(r, name=part))
    out = pd.DataFrame(rows).fillna(0)
    print(f'\n{name}: composizione di test e delle validation dei 5 fold')
    display(out[['n', 'lesion_mm3_mediana'] + [c for c in out.columns
                if c not in ('n', 'lesion_mm3_mediana')][:8]])

## Step 8 - Materializzazione nel formato nnU-Net raw

nnU-Net v2 richiede questo layout, con il canale codificato nel suffisso `_0000`, `_0001`, ...:

```
Dataset501_WMH/
  imagesTr/Utrecht_0_0000.nii.gz      # FLAIR   (+ _0001 se WMH_USE_T1)
  labelsTr/Utrecht_0.nii.gz
  imagesTs/...                        # nostro test interno
  labelsTs/...                        # label del test interno (nnU-Net le ignora, servono a noi)
  dataset.json
```

**Immagini**: symlink assoluti ai file originali. Non occupano spazio, non modificano i dati, e nnU-Net
li legge normalmente. Con `MATERIALIZE_IMAGES = 'copy'` si ottengono copie vere (circa 14 GB in totale).

**Label**: sempre **riscritte**, perché devono essere normalizzate:

- binarizzate con `> 0` (ATLAS) o rimappate secondo `WMH_LABEL2_POLICY` (WMH);
- convertite a `uint8`, che è ciò che nnU-Net si aspetta per le segmentazioni;
- salvate con l'**affine dell'immagine**, non della label, così le due geometrie sono bit-identiche e
  nnU-Net non emette warning di mismatch.

Il test set interno finisce in `imagesTs`/`labelsTs`: **non** entra in `imagesTr`, quindi resta fuori dal
fingerprint e dal planning.

In [ ]:
def link_or_copy(src, dst, mode=None):
    mode = mode or MATERIALIZE_IMAGES
    if dst.is_symlink() or dst.exists():
        dst.unlink()
    if mode == 'symlink':
        dst.symlink_to(src.resolve())
    else:
        shutil.copy2(src, dst)


def write_label(case, dst, policy=WMH_LABEL2_POLICY):
    """Riscrive la label normalizzata, usando l'affine dell'immagine."""
    img = nib.load(str(case['image']), mmap=True)
    lab = nib.load(str(case['label']))
    data = np.asanyarray(lab.dataobj)
    out = np.zeros(data.shape, dtype=np.uint8)

    if case['dataset'] == 'ATLAS':
        out[data > 0] = 1                      # robusto alle maschere codificate 0/0.01
    else:
        r = np.rint(data).astype(np.int16)
        out[r == 1] = 1
        if policy == 'merge':
            out[r == 2] = 1
        elif policy == 'ignore':
            out[r == 2] = 2                    # ignore label di nnU-Net
        # 'exclude': la label 2 resta 0

    new = nib.Nifti1Image(out, img.affine)
    new.header.set_zooms(tuple(float(v) for v in img.header.get_zooms()[:3]))
    new.set_data_dtype(np.uint8)
    nib.save(new, str(dst))


def channels_for(case):
    ch = [case['image']]
    if case['dataset'] == 'WMH' and WMH_USE_T1:
        assert case['image2'] is not None, f"{case['case_id']}: T1 richiesto ma assente"
        ch.append(case['image2'])
    return ch


def materialize(name):
    ds_dir = RAW_OUT / f'Dataset{DATASET_IDS[name]:03d}_{name}'
    for sub in ('imagesTr', 'labelsTr', 'imagesTs', 'labelsTs'):
        (ds_dir / sub).mkdir(parents=True, exist_ok=True)

    by_id = {c['case_id']: c for c in CASES[name]}
    s = SPLITS[name]
    plan = [(cid, 'Tr') for cid in s['development']] + [(cid, 'Ts') for cid in s['internal_test']]

    t0 = time.time()
    for i, (cid, part) in enumerate(plan, 1):
        c = by_id[cid]
        for ch_idx, src in enumerate(channels_for(c)):
            link_or_copy(src, ds_dir / f'images{part}' / f'{cid}_{ch_idx:04d}.nii.gz')
        write_label(c, ds_dir / f'labels{part}' / f'{cid}.nii.gz')
        if i % 100 == 0 or i == len(plan):
            print(f'  {name}: {i}/{len(plan)} ({time.time()-t0:.0f}s)', flush=True)

    # test ufficiale ATLAS senza label: cartella separata, fuori dal flusso nnU-Net
    if s['official_test_unlabeled']:
        off = ds_dir / 'imagesTs_official'
        off.mkdir(exist_ok=True)
        for cid in s['official_test_unlabeled']:
            for ch_idx, src in enumerate(channels_for(by_id[cid])):
                link_or_copy(src, off / f'{cid}_{ch_idx:04d}.nii.gz')
        print(f'  {name}: {len(s["official_test_unlabeled"])} immagini ufficiali senza label '
              f'-> imagesTs_official')
    return ds_dir


DS_DIRS = {}
for name in ('WMH', 'ATLAS'):
    print(f'--- materializzazione {name} ---')
    DS_DIRS[name] = materialize(name)
    print('  ->', DS_DIRS[name])

## Step 9 - `dataset.json` e `splits_final.json`

`dataset.json` dichiara i canali e le label. Il **nome del canale conta**: nnU-Net sceglie la
normalizzazione in base ad esso. Per le modalità MR (`FLAIR`, `T1`) applica z-score sul foreground,
che è ciò che vogliamo; solo `CT` attiva la normalizzazione basata su percentili di Hounsfield.

`splits_final.json` va in `nnUNet_preprocessed/DatasetXXX/`. nnU-Net lo genera da solo **solo se non
esiste**: scrivendolo ora, i nostri fold stratificati vengono rispettati.

Infine creiamo i symlink da `nnUNet/nnUNet_raw/` verso i dataset in `data/processed/`, così la
convenzione del progetto e la variabile d'ambiente `nnUNet_raw` continuano a funzionare senza duplicare
nulla.

In [ ]:
def write_dataset_json(name, ds_dir):
    if name == 'WMH':
        channels = {'0': 'FLAIR'}
        if WMH_USE_T1:
            channels['1'] = 'T1'
        labels = {'background': 0, 'wmh': 1}
        if WMH_LABEL2_POLICY == 'ignore':
            labels['ignore'] = 2
    else:
        channels = {'0': 'T1'}
        labels = {'background': 0, 'lesion': 1}

    n_train = len(list((ds_dir / 'labelsTr').glob('*.nii.gz')))
    meta = dict(name=f'Dataset{DATASET_IDS[name]:03d}_{name}',
                description=f'{name} lesion segmentation, preparato da notebooks/dataset_preparation.ipynb',
                channel_names=channels,
                labels=labels,
                numTraining=n_train,
                file_ending='.nii.gz',
                dataset_preparation=dict(
                    seed=SEED, n_folds=N_FOLDS, test_fraction=TEST_FRACTION[name],
                    stratified_on=SPLITS[name]['stratified_on'],
                    wmh_label2_policy=WMH_LABEL2_POLICY if name == 'WMH' else None,
                    predicted_target_spacing=np.round(PLAN[name]['target_spacing'], 4).tolist(),
                ))
    (ds_dir / 'dataset.json').write_text(json.dumps(meta, indent=2))
    return meta


def write_splits_final(name, ds_dir):
    out_dir = NNUNET_PREPROCESSED / ds_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / 'splits_final.json'
    path.write_text(json.dumps(SPLITS[name]['folds'], indent=2))
    return path


for name, ds_dir in DS_DIRS.items():
    meta = write_dataset_json(name, ds_dir)
    sp = write_splits_final(name, ds_dir)
    link = NNUNET_RAW / ds_dir.name
    if link.is_symlink() or link.exists():
        link.unlink() if link.is_symlink() else shutil.rmtree(link)
    link.symlink_to(ds_dir.resolve())
    print(f'{name}: dataset.json numTraining={meta["numTraining"]}  channels={meta["channel_names"]}  '
          f'labels={meta["labels"]}')
    print(f'      splits_final.json -> {sp}')
    print(f'      symlink           -> {link}')

## Step 10 - QC finale

Controlli di integrità sul prodotto finito, più un'ispezione visiva. Se qualcosa è incoerente qui,
il preprocessing nnU-Net fallirebbe (o peggio, girerebbe su dati sbagliati).

In [ ]:
n_ch = {'WMH': 2 if WMH_USE_T1 else 1, 'ATLAS': 1}

for name, ds_dir in DS_DIRS.items():
    labels_tr = sorted(p.name[:-7] for p in (ds_dir / 'labelsTr').glob('*.nii.gz'))
    labels_ts = sorted(p.name[:-7] for p in (ds_dir / 'labelsTs').glob('*.nii.gz'))
    assert labels_tr == sorted(SPLITS[name]['development']), f'{name}: labelsTr non coerente con lo split'
    assert labels_ts == sorted(SPLITS[name]['internal_test']), f'{name}: labelsTs non coerente con lo split'

    for part, ids in (('Tr', labels_tr), ('Ts', labels_ts)):
        for cid in ids:
            for k in range(n_ch[name]):
                f = ds_dir / f'images{part}' / f'{cid}_{k:04d}.nii.gz'
                assert f.exists(), f'{name}: manca {f.name}'
                assert f.stat().st_size > 0, f'{name}: file vuoto {f.name}'

    # geometria e valori sulle label riscritte
    bad, vals = [], set()
    for cid in labels_tr[:50] + labels_ts[:20]:
        part = 'Tr' if cid in set(labels_tr) else 'Ts'
        lab = nib.load(str(ds_dir / f'labels{part}' / f'{cid}.nii.gz'))
        img = nib.load(str(ds_dir / f'images{part}' / f'{cid}_0000.nii.gz'), mmap=True)
        if lab.shape[:3] != img.shape[:3] or not np.allclose(lab.affine, img.affine, atol=1e-4):
            bad.append(cid)
        vals |= set(np.unique(np.asanyarray(lab.dataobj)).astype(int).tolist())
        assert lab.get_data_dtype() == np.uint8, f'{name}/{cid}: dtype label non uint8'
    assert not bad, f'{name}: geometria label/immagine incoerente in {bad[:5]}'

    size_mb = sum(f.stat().st_size for f in ds_dir.rglob('*') if f.is_file() and not f.is_symlink()) / 1e6
    print(f'{name}: {len(labels_tr)} training + {len(labels_ts)} test  |  valori label {sorted(vals)}  |  '
          f'occupazione reale {size_mb:.0f} MB (modalita {MATERIALIZE_IMAGES})')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, name in enumerate(['WMH', 'ATLAS']):
    ds_dir = DS_DIRS[name]
    df = MANIFEST[name].set_index('case_id')
    dev = SPLITS[name]['development']
    # un caso a lesione piccola, uno mediana, uno grande: il caso peggiore e' quello piccolo
    sel = df.loc[dev].sort_values('lesion_mm3')
    picks = [sel.index[0], sel.index[len(sel) // 2], sel.index[-1]]
    for ax, cid in zip(axes[row], picks):
        img = nib.load(str(ds_dir / 'imagesTr' / f'{cid}_0000.nii.gz')).get_fdata(dtype=np.float32)
        lab = np.asanyarray(nib.load(str(ds_dir / 'labelsTr' / f'{cid}.nii.gz')).dataobj)
        z = int(np.argmax(lab.sum(axis=(0, 1))))          # fetta con piu' lesione
        ax.imshow(np.rot90(img[:, :, z]), cmap='gray',
                  vmin=np.percentile(img, 1), vmax=np.percentile(img, 99.5))
        ax.imshow(np.rot90(np.ma.masked_where(lab[:, :, z] == 0, lab[:, :, z])),
                  cmap='autumn', alpha=.6, vmin=0, vmax=2)
        ax.set_title(f'{name} {cid}\n{df.loc[cid,"lesion_mm3"]:.0f} mm³, z={z}', fontsize=9)
        ax.axis('off')
plt.suptitle('QC visivo: overlay label sulle immagini materializzate (lesione minima / mediana / massima)')
plt.tight_layout()
plt.savefig(QC_DIR / 'overlay_qc.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 11 - Passi successivi

Le variabili d'ambiente e i comandi nnU-Net da eseguire dal terminale. Il planning e il preprocessing
sono la fase che decide davvero la risoluzione: la cella sotto stampa la target spacing prevista in
questo notebook, da confrontare con quella nel `nnUNetPlans.json` generato dal planner.

Per **forzare** una target spacing diversa da quella scelta automaticamente (ad esempio per confrontare
un modello a 3 mm con uno isotropico a 1 mm su WMH) si usa
`nnUNetv2_plan_experiment -overwrite_target_spacing z y x`, che produce un piano alternativo da
affiancare al default senza sovrascriverlo.

In [ ]:
env = {'nnUNet_raw': NNUNET_RAW, 'nnUNet_preprocessed': NNUNET_PREPROCESSED,
       'nnUNet_results': NNUNET_RESULTS}
print('# variabili d\'ambiente')
for k, v in env.items():
    print(f'export {k}="{v}"')

print('\n# verifica, planning e preprocessing')
for name, ds_dir in DS_DIRS.items():
    did = DATASET_IDS[name]
    print(f'nnUNetv2_plan_and_preprocess -d {did} --verify_dataset_integrity   # {name}')

print('\n# training (fold definiti dai nostri splits_final.json)')
for name in DS_DIRS:
    print(f'for f in 0 1 2 3 4; do nnUNetv2_train {DATASET_IDS[name]} 3d_fullres $f; done   # {name}')

print('\n# target spacing prevista da questo notebook (da confrontare con nnUNetPlans.json)')
for name in DS_DIRS:
    ts = np.round(PLAN[name]['target_spacing'], 4).tolist()
    print(f'  {name}: {ts} mm   resampling separato su z = {PLAN[name]["do_separate_z"]}')

import shutil as _sh
free_gb = _sh.disk_usage(PROJECT).free / 1e9
print(f'\n# spazio libero sul filesystem del progetto: {free_gb:.0f} GB')
if free_gb < 200:
    print('  ATTENZIONE: il preprocessing nnU-Net genera decine di GB per dataset e configurazione.')
    print('  Valutare di puntare nnUNet_preprocessed su un filesystem piu capiente.')

## Step 12 - Report riproducibile

`reports/` contiene report rigenerabili, non scritti a mano. Questa cella scrive
`reports/dataset_preparation.md` e `.json` a partire dagli oggetti in memoria, così il contenuto
del report non può divergere da ciò che il notebook ha effettivamente prodotto.

In [ ]:
from datetime import date

REPORTS = PROJECT / 'reports'
REPORTS.mkdir(exist_ok=True)

payload, lines = {}, []
lines += ['# Dataset preparation', '',
          f'Generato da `notebooks/dataset_preparation.ipynb` il {date.today().isoformat()}. '
          f'Seed `{SEED}`, {N_FOLDS} fold.', '',
          'Report dei dati **appaiati** (immagine + label per caso), a differenza di '
          '`dataset_exploration.md` che e\' un inventario grezzo di tutti i file.', '']

for name in ('WMH', 'ATLAS'):
    df, s, p = MANIFEST[name], SPLITS[name], PLAN[name]
    tr = df[df.source_split == 'train']
    ds_dir = DS_DIRS[name]

    lines += [f'## {name} (Dataset{DATASET_IDS[name]:03d})', '',
              f"- Sorgente: `{CFG['datasets'][name]['path']}`",
              f'- Casi appaiati: **{len(tr)}**  |  gruppi: **{tr.group.nunique()}**  '
              f'({"sito/scanner" if name == "WMH" else "coorte Rxxx"})',
              f'- Immagini senza label (test ufficiale): **{len(s["official_test_unlabeled"])}**',
              f'- Orientamento: `{"/".join(sorted(df.orientation.unique()))}`',
              f'- Spacing distinti: **{tr[["sx","sy","sz"]].round(4).drop_duplicates().shape[0]}**', '']

    def res_row(label, sub):
        return (f'| {label} | {len(sub)} | {sub.sx.median():.4g} x {sub.sy.median():.4g} x '
                f'{sub.sz.median():.4g} | {sub.nx.median():.0f}x{sub.ny.median():.0f}x'
                f'{sub.nz.median():.0f} | {sub.voxel_vol_mm3.median():.3f} | '
                f'{sub.anisotropy.median():.2f} |')

    uniform = tr[['sx', 'sy', 'sz']].round(4).drop_duplicates().shape[0] == 1
    lines += ['### Risoluzione spaziale', '']
    if uniform:
        lines += [f'Risoluzione **uniforme** su tutti i {tr.group.nunique()} gruppi: '
                  'una sola riga riassuntiva.', '']
    lines += ['| Gruppo | n | spacing (mm) | shape mediana | voxel mm3 | anisotropia |',
              '| --- | --- | --- | --- | --- | --- |']
    if uniform:
        lines.append(res_row(f'tutti ({tr.group.nunique()} gruppi)', tr))
    else:
        for g, sub in tr.groupby('group'):
            lines.append(res_row(g, sub))

    d = p['diag']
    lines += ['', '### Target spacing prevista (regola nnU-Net v2)', '',
              f"- spacing mediano: `{d['spacing_mediano']}` mm, shape mediana `{d['shape_mediana']}`",
              f"- asse peggiore: `{'xyz'[d['asse_peggiore']]}`",
              f"- condizione spacing: `{d['cond_spacing']}`",
              f"- condizione voxel: `{d['cond_voxel']}`",
              f"- override anisotropico applicato: **{d['override_applicato']}**",
              f"- **target spacing: `{np.round(p['target_spacing'], 4).tolist()}` mm**",
              f"- resampling separato lungo z (anisotropia > {ANISO_THRESHOLD}): "
              f"**{p['do_separate_z']}**", '']

    vox_after = tr.lesion_mm3 / float(np.prod(p['target_spacing']))
    lines += ['### Carico lesionale', '',
              f'- volume lesione (mm³): min {tr.lesion_mm3.min():.0f}, p10 {tr.lesion_mm3.quantile(.1):.0f}, '
              f'mediana {tr.lesion_mm3.median():.0f}, p90 {tr.lesion_mm3.quantile(.9):.0f}, '
              f'max {tr.lesion_mm3.max():.0f}',
              f'- casi con lesione vuota: {int((tr.n_lesion_vox == 0).sum())}',
              f'- casi sotto 100 voxel alla risoluzione target: {int((vox_after < 100).sum())} / {len(tr)}',
              '', '### Label', '',
              f'- valori osservati: `{tr[tr.label_values != ""].label_values.value_counts().to_dict()}`']
    if name == 'WMH':
        lines += [f'- label 2 ("altra patologia") presente in **{int((tr.n_other_vox > 0).sum())}/{len(tr)}** '
                  f'casi, volume max {tr.other_mm3.max():.0f} mm³',
                  f"- politica applicata: `{WMH_LABEL2_POLICY}`"]
    else:
        anomali = tr[(tr.label_values != '') & (~tr.label_values.isin(['0/1', '0']))]
        lines += ['- le due varianti `_mask` / `_mask_value2` condividono lo stesso supporto binario; '
                  'usata `_mask`',
                  f'- codifiche anomale: {anomali.case_id.tolist() or "nessuna"} '
                  '(binarizzazione con `> 0`)']

    lines += ['', '### Split', '',
              f"- stratificazione: `{s['stratified_on']}`",
              f"- development (5-fold CV): **{len(s['development'])}**  |  "
              f"test interno: **{len(s['internal_test'])}**  |  "
              f"test ufficiale senza label: **{len(s['official_test_unlabeled'])}**",
              '- fold (train/val): ' + ', '.join(f"{len(f['train'])}/{len(f['val'])}" for f in s['folds']),
              f'- output: `{ds_dir.relative_to(PROJECT)}`', '']

    payload[name] = dict(
        n_cases=int(len(tr)), n_groups=int(tr.group.nunique()),
        orientations=sorted(df.orientation.unique()),
        target_spacing=np.round(p['target_spacing'], 4).tolist(),
        target_spacing_diagnostics=d, do_separate_z=p['do_separate_z'],
        lesion_mm3=dict(min=float(tr.lesion_mm3.min()), median=float(tr.lesion_mm3.median()),
                       max=float(tr.lesion_mm3.max())),
        n_under_100_voxels=int((vox_after < 100).sum()),
        label_values=tr[tr.label_values != ''].label_values.value_counts().to_dict(),
        wmh_label2_policy=WMH_LABEL2_POLICY if name == 'WMH' else None,
        split=dict(stratified_on=s['stratified_on'], development=len(s['development']),
                   internal_test=len(s['internal_test']),
                   official_test_unlabeled=len(s['official_test_unlabeled']),
                   folds=[dict(train=len(f['train']), val=len(f['val'])) for f in s['folds']]),
        dataset_dir=str(ds_dir.relative_to(PROJECT)))

lines += ['## Riproduzione', '',
          'Eseguire `notebooks/dataset_preparation.ipynb` dall\'inizio alla fine. I manifest sono in '
          'cache in `data/processed/manifests/`: per ricalcolarli da zero impostare '
          '`FORCE_RECOMPUTE = True` nella cella di configurazione.', '']

(REPORTS / 'dataset_preparation.md').write_text('\n'.join(lines))
(REPORTS / 'dataset_preparation.json').write_text(
    json.dumps(dict(generated=date.today().isoformat(), seed=SEED, n_folds=N_FOLDS,
                    datasets=payload), indent=2) + '\n')
print('scritti:')
print(' ', REPORTS / 'dataset_preparation.md')
print(' ', REPORTS / 'dataset_preparation.json')